In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
import torch
print(torch.cuda.is_available())

False


In [27]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [9]:
DATASET_PATH = "dataset"

train_path = os.path.join(DATASET_PATH, "train/images")
val_path = os.path.join(DATASET_PATH, "valid/images")
test_path = os.path.join(DATASET_PATH, "test/images")

print("Train images:", len(os.listdir(train_path)))
print("Validation images:", len(os.listdir(val_path)))
print("Test images:", len(os.listdir(test_path)))

Train images: 140
Validation images: 40
Test images: 20


In [10]:
image_files = os.listdir(train_path)
sample_images = random.sample(image_files, 4)

plt.figure(figsize=(10,8))

for i,img_name in enumerate(sample_images):
    
    img = cv2.imread(os.path.join(train_path, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.title(img_name)
    plt.axis("off")

plt.show()

<Figure size 1000x800 with 4 Axes>

In [11]:
yaml_content = f"""
path: {DATASET_PATH}

train: train/images
val: valid/images
test: test/images

names:
  0: pedestrian
  1: car
"""

with open("dataset.yaml","w") as f:
    f.write(yaml_content)

print("dataset.yaml created")

dataset.yaml created


In [17]:
model = YOLO("yolov8n.pt")

True


In [29]:
results = model.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=416,
    batch=4,
    device="cpu",
    workers=0,
    name="pedestrian_car_detection"
)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.240  Python-3.12.5 torch-2.7.1+cu118 CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=pedestrian_car_detection8, nbs=64, nms=Fa

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
metrics = model.val()

print("Precision:", metrics.box.precision)
print("Recall:", metrics.box.recall)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

In [ ]:
model.predict(
    source=test_path,
    conf=0.25,
    save=True
)

In [ ]:
prediction_dir = "runs/detect/predict"

pred_images = os.listdir(prediction_dir)[:4]

plt.figure(figsize=(10,8))

for i,img_name in enumerate(pred_images):

    img = cv2.imread(os.path.join(prediction_dir,img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()

In [ ]:
from IPython.display import Image

Image("runs/detect/pedestrian_car_detection/results.png")